# Chess.com Data Fetcher
**TP InfoVis · ITBA Maestría en Ciencia de Datos**

Este notebook descarga las partidas de Chess.com vía API pública y genera 4 CSVs listos para Flourish, guardados directamente en Google Drive.

## 1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuración

In [ ]:
import os

USERNAME   = "sugeema"
OUTPUT_DIR = "/content/drive/MyDrive/InfoVis-TP/chess_data"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Carpeta de salida: {OUTPUT_DIR}")

## 3. Descargar partidas de la API

In [ ]:
import requests
import time
from datetime import datetime, timezone

HEADERS = {"User-Agent": "InfoVis-ITBA-TP/1.0"}

MODALIDAD_LABEL = {
    "bullet": "Bullet",
    "blitz":  "Blitz",
    "rapid":  "Rapid",
    "daily":  "Daily",
}

def get_archives(username):
    url = f"https://api.chess.com/pub/player/{username}/games/archives"
    r = requests.get(url, headers=HEADERS)
    r.raise_for_status()
    return r.json().get("archives", [])

def get_games(archive_url):
    r = requests.get(archive_url, headers=HEADERS)
    r.raise_for_status()
    return r.json().get("games", [])

def parse_game(game, username):
    time_class = game.get("time_class", "")
    end_time   = game.get("end_time")
    rules      = game.get("rules", "chess")

    if rules != "chess" or end_time is None:
        return None

    white = game.get("white", {})
    black = game.get("black", {})

    if white.get("username", "").lower() == username.lower():
        color, my_data, opp_data = "white", white, black
    elif black.get("username", "").lower() == username.lower():
        color, my_data, opp_data = "black", black, white
    else:
        return None

    result_raw = my_data.get("result", "")
    rating     = my_data.get("rating")
    opp_rating = opp_data.get("rating")

    if result_raw == "win":
        result = "Win"
    elif result_raw in ("checkmated","timeout","resigned","lose","abandoned"):
        result = "Loss"
    elif result_raw in ("agreed","repetition","stalemate","insufficient",
                        "50move","timevsinsufficient","drawbyrepetition"):
        result = "Draw"
    else:
        return None

    if rating is None:
        return None

    dt = datetime.fromtimestamp(end_time, tz=timezone.utc)

    return {
        "date":        dt.strftime("%Y-%m-%d"),
        "datetime":    dt.strftime("%Y-%m-%d %H:%M"),
        "year":        dt.year,
        "month":       dt.strftime("%Y-%m"),
        "weekday":     dt.strftime("%A"),
        "weekday_num": dt.weekday(),
        "hour":        dt.hour,
        "time_class":  time_class,
        "modalidad":   MODALIDAD_LABEL.get(time_class, time_class.capitalize()),
        "color":       color,
        "result":      result,
        "rating":      int(rating),
        "opp_rating":  int(opp_rating) if opp_rating else None,
        "time_control": game.get("time_control", ""),
    }

# ── Descargar todo ──
print(f"Descargando partidas de '{USERNAME}'...\n")
archives = get_archives(USERNAME)
print(f"Meses disponibles: {len(archives)}")

all_games = []
for i, url in enumerate(archives):
    label = url.split("/")[-2] + "/" + url.split("/")[-1]
    games  = get_games(url)
    parsed = [parse_game(g, USERNAME) for g in games]
    parsed = [g for g in parsed if g is not None]
    all_games.extend(parsed)
    print(f"  [{i+1:02d}/{len(archives)}] {label} → {len(parsed)} partidas")
    time.sleep(0.3)

print(f"\nTotal: {len(all_games)} partidas")

## 4. Generar CSVs y guardar en Drive

In [ ]:
import csv
from collections import defaultdict

games_sorted = sorted(all_games, key=lambda x: x["datetime"])

# ── CSV 1: Evolución del rating ──────────────────────────────────────────
path1 = os.path.join(OUTPUT_DIR, "01_rating_evolution.csv")
fields1 = ["date","datetime","modalidad","rating","opp_rating","result","color"]
with open(path1, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fields1)
    w.writeheader()
    for g in games_sorted:
        w.writerow({k: g[k] for k in fields1})
print(f"✓ 01_rating_evolution.csv  ({len(games_sorted)} filas)")

# ── CSV 2: W/L/D por mes ────────────────────────────────────────────────
counts = defaultdict(lambda: {"Win":0,"Loss":0,"Draw":0,"total":0})
for g in all_games:
    key = (g["month"], g["modalidad"])
    counts[key][g["result"]] += 1
    counts[key]["total"]     += 1

path2 = os.path.join(OUTPUT_DIR, "02_results_by_month.csv")
with open(path2, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["month","modalidad","win","loss","draw","total","win_pct"])
    for (month, mod), v in sorted(counts.items()):
        win_pct = round(v["Win"] / v["total"] * 100, 1) if v["total"] else 0
        w.writerow([month, mod, v["Win"], v["Loss"], v["Draw"], v["total"], win_pct])
print(f"✓ 02_results_by_month.csv  ({len(counts)} filas)")

# ── CSV 3: Heatmap día × hora ───────────────────────────────────────────
heatmap = defaultdict(int)
for g in all_games:
    heatmap[(g["weekday_num"], g["weekday"], g["hour"])] += 1

DAYS = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
path3 = os.path.join(OUTPUT_DIR, "03_activity_heatmap.csv")
with open(path3, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["weekday_num","weekday","hour","games"])
    for day_num, day_name in enumerate(DAYS):
        for hour in range(24):
            w.writerow([day_num, day_name, hour, heatmap[(day_num, day_name, hour)]])
print(f"✓ 03_activity_heatmap.csv  (168 filas)")

# ── CSV 4: Rendimiento por color ────────────────────────────────────────
color_counts = defaultdict(lambda: {"Win":0,"Loss":0,"Draw":0,"total":0})
for g in all_games:
    key = (g["color"], g["modalidad"])
    color_counts[key][g["result"]] += 1
    color_counts[key]["total"]     += 1

path4 = os.path.join(OUTPUT_DIR, "04_color_performance.csv")
with open(path4, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["color","modalidad","win","loss","draw","total","win_pct"])
    for (color, mod), v in sorted(color_counts.items()):
        win_pct = round(v["Win"] / v["total"] * 100, 1) if v["total"] else 0
        w.writerow([color, mod, v["Win"], v["Loss"], v["Draw"], v["total"], win_pct])
print(f"✓ 04_color_performance.csv  ({len(color_counts)} filas)")

print(f"\n✅ Todos los archivos guardados en:\n   {OUTPUT_DIR}")

## 5. Preview de los datos

In [ ]:
import pandas as pd

df = pd.read_csv(path1)
print(f"Rating evolution: {len(df)} partidas")
print(f"Modalidades: {df['modalidad'].unique()}")
print(f"Período: {df['date'].min()} → {df['date'].max()}")
print(f"Rating actual (último): {df['rating'].iloc[-1]}")
print()
df.tail(5)